# Multi-Query Retrieval for RAG

A common weakness of plain RAG is that distance-based similarity search is sensitive to exactly how a question is phrased — two questions that mean the same thing can retrieve very different chunks. **Multi-query retrieval** works around this by asking an LLM to rewrite the user's question into several different phrasings, retrieving documents for *each* variant, and then merging the results into one deduplicated set of context before generation.

This notebook builds that pipeline in four stages:

1. **Indexing** — load a Wikipedia page, split it into chunks, and embed it into a vector store.
2. **Query generation** — use an LLM to generate multiple rephrasings of the original question.
3. **Multi-retrieval & deduplication** — retrieve documents for every rephrased query and merge them into a single unique set.
4. **Generation** — answer the original question, grounded in the combined, deduplicated context.

As a worked example, the pipeline indexes the Wikipedia article on [FIFA](https://en.wikipedia.org/wiki/FIFA) and answers *"What is FIFA?"*

### Requirements
```bash
pip install langchain langchain-community langchain-text-splitters langchain-ollama langchain-huggingface chromadb sentence-transformers bs4
```
You'll also need [Ollama](https://ollama.com/) installed locally with the `phi3` model pulled:
```bash
ollama pull phi3
```

## 1. Indexing

### 1.1 Load the document

We use `WebBaseLoader` to scrape the target Wikipedia page, restricting extraction to the main article body (`div#mw-content-text`) via a `bs4.SoupStrainer` so we don't pull in the sidebar, navigation, references list, or other boilerplate.

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Retrieval-augmented_generation",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            id="mw-content-text"
        )
    ),
)
source_docs = loader.load()
source_docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation'}, page_content='Type of information retrieval using LLMs\nRetrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources.[1][2] With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM\'s pre-existing training data.[3] This allows LLMs to use domain-specific and/or updated information that is not available in the training data.[3] For example, this enables LLM-based chatbots to access internal company data or generate responses based on authoritative sources. The technique was first proposed in 2020 and has since become a widely adopted approach in modern AI systems.\nRAG improves LLMs by incorporating information retrieval before generating responses.[4] Unlike LLMs that rely on static training data, RAG pulls

### 1.2 Split the document into chunks

`RecursiveCharacterTextSplitter` breaks the page into overlapping chunks small enough to embed and retrieve individually, while keeping some continuity across chunk boundaries via `chunk_overlap`.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunker = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
chunks = chunker.split_documents(source_docs)

print(f"Split into {len(chunks)} chunks")
chunks[0]

Split into 106 chunks


Document(metadata={'source': 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation'}, page_content='Type of information retrieval using LLMs')

### 1.3 Embed chunks and build a vector store

Each chunk is embedded with a local `sentence-transformers` model and stored in a **Chroma** vector store. The retriever wraps the vector store and returns the top matching chunks for a given query.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embed)
retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2252.39it/s]


## 2. Generating Multiple Queries

Instead of retrieving with only the original question, we prompt the LLM to produce five alternative phrasings. Each rephrasing approaches the question from a slightly different angle, which widens the net of relevant chunks the retriever can find.

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}
"""

multiple_prompts = ChatPromptTemplate.from_template(template)
llm = ChatOllama(model="phi3", temperature=0)

generate_queries = multiple_prompts | llm | StrOutputParser() | (lambda x: x.strip("\n"))
# generate_queries.invoke({"question": "What is retrieval-augmented generation?"})

## 3. Retrieval & Deduplication

Each generated query is run through the retriever (`retriever.map()` applies the retriever to every query in the list), producing a list of document lists — one per query. `get_unique_docs` flattens that structure and removes duplicates (documents retrieved by more than one query variant), using LangChain's `dumps`/`loads` to make each `Document` hashable for deduplication via a `set`.

In [10]:
from langchain_core.load import dumps, loads

def get_unique_docs(nested_docs: list[list]):
    flattened_list = [dumps(doc) for sublist in nested_docs for doc in sublist]
    unique_docs = list(set(flattened_list))
    return [loads(doc) for doc in unique_docs]

retrieval_chain = generate_queries | retriever.map() | get_unique_docs

## 4. Generation

Finally, the original question is answered using the combined, deduplicated context gathered from all five query variants — giving the LLM a broader and more robust set of evidence than a single-query retrieval would.

In [11]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_chain.invoke({"question": "What is retrieval-augmented generation?"})

C:\Users\chett\AppData\Local\Temp\ipykernel_13960\202648930.py:6: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


"Retrieval-augmented generation is a method in natural language processing (NLP) that enhances the capabilities of large language models (LLMs) by integrating external knowledge sources. This approach combines the generative power of LLMs with the ability to retrieve relevant information from vast datasets or knowledge bases, such as databases or the internet. The goal is to improve the accuracy, relevance, and factual correctness of the generated responses by providing the LLM with additional context and information that it might not have learned during its training phase.\n\nThe process typically involves prompting the LLM with a query or a task, and then using retrieval mechanisms to fetch pertinent information from external sources. This information is then incorporated into the LLM's internal representation, allowing it to generate more informed and contextually appropriate responses. Retrieval-augmented generation can help reduce hallucinations, where AI models generate plausible